# Laboratorio 4: Detección de Movimiento y Seguimiento de Objetos

### Objetivo
En esta práctica, aprenderás a implementar algoritmos de detección de movimiento mediante sustracción de fondo y flujo óptico, y explorarás el filtro de Kalman para el seguimiento de objetos.

## Materiales
- **Python 3.8+**
- **OpenCV**: Puedes instalarlo con `pip install opencv-python`
- **Dataset de video**: Se usará un archivo de video o la cámara en tiempo real para probar los métodos de detección de movimiento.

In [2]:
import cv2
import os
import numpy as np

## Apartado C: Sustracción de Fondo

### Tarea C.1: Carga de Video
Carga un video en el cual se detectarán objetos en movimiento. Puedes utilizar
un video local o la cámara en tiempo real.

In [3]:
#TODO: Use VideoCapture from OpenCV to read the video (visiontraffic.avi)
videopath = "../data/partC/visiontraffic.avi" # Path to the video file

def read_video(videopath):
    cap = cv2.VideoCapture(videopath )  # Complete this line to read the video file

    #TODO: Check if the video was successfully opened
    if not cap.isOpened():
        print('Error: Could not open the video file')
        os._exit(0)

    #TODO: Get the szie of frames and the frame rate of the video
    frame_width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
    frame_height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
    frame_rate = cap.get(cv2.CAP_PROP_FPS)

    #TODO: Use a loop to read the frames of the video and store them in a list
    frames = []
    while True:
        ret, frame = cap.read()
        if not ret:
            break
        frames.append(frame)
    cap.release()
    return frames, frame_width, frame_height, frame_rate


frames, frame_width, frame_height, frame_rate = read_video(videopath)
print(f"Número de frames: {len(frames)}")
print(f"Resolución: {frame_width}x{frame_height}")
print(f"Frame rate: {frame_rate}")



Número de frames: 531
Resolución: 640x360
Frame rate: 29.97002997002997


### Tarea C.2: Sustración de Fondo mediante diferencia de frames
Realiza una sustracción de fondo mediante diferencia de frames, para ello guarda
un frame con el fondo estático y úsalo como frame de referencia de fondo.

In [4]:
#TODO:  Show the frames to select the reference frame, press 'n' to move to the next frame and 's' to select the frame
for i, frame in enumerate(frames):
    # Show the frame
    cv2.imshow('Video', frame)
    # Wait for the key
    key = cv2.waitKey(0)
    # If the key is 'n' continue to the next frame
    if key == ord('n'):
        continue
    # If the key is 's' select the frame as the reference frame
    elif key == ord('s'):
        # Copy the frame to use it as a reference
        reference_frame = frame
        # Convert the reference frame to grayscale
        reference_frame = cv2.cvtColor(reference_frame, cv2.COLOR_BGR2GRAY)
        print('Frame {} selected as reference frame'.format(i))
        break

cv2.destroyAllWindows()
diffs=[]
#TODO: Compute the difference between the reference frame and the rest of the frames and show the difference
for frame in frames:
    # Convert the frame to grayscale
    frame = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)
    diff = cv2.absdiff(frame, reference_frame)
    diffs.append(diff)
    cv2.imshow('Diferencia', diff)
    key = cv2.waitKey(1)
    if key == ord('q'):
        break
cv2.imshow('Diferencia', diffs[200])
key = cv2.waitKey(0)
cv2.destroyAllWindows()

Frame 8 selected as reference frame


### Tarea AC.3: Configuración de la sustración de fondo con GMM
Configura el sustractor de fondo usando el modelo de mezcla de gaussianas
adaptativas (MOG2).

In [5]:
#TODO: Use MOG2 to detect the moving objects in the video

history = 200  # Number of frames to use to build the background model
varThreshold = 16  # Threshold to detect the background
detectShadows = True    # If True the algorithm detects the shadows

# Create the MOG2 object
mog2 = cv2.createBackgroundSubtractorMOG2(history=history,varThreshold=varThreshold, detectShadows=detectShadows)


### Tarea C.4: Aplicación de la Sustracción de Fondo

Aplica la sustracción de fondo en cada frame para extraer los objetos en movimiento.

In [6]:
#TODO: Use a loop to detect the moving objects in the video using the MOG2 algorithm and 
# save a video storing the parameters at the name of the file

# Create a folder to store the videos
output_folder = "../data/partC/results_C4"
if not os.path.exists(output_folder):
    os.makedirs(output_folder)

videoname = f'output_{history}_{varThreshold}_{"shadows" if detectShadows else "noshadows"}.avi' # Name of the output video file with the parameters
videopath = os.path.join(output_folder, videoname)

# Create a VideoWriter object to save the video
fourcc = cv2.VideoWriter_fourcc(*'XVID') # Codec to use
frame_size = (frame_width, frame_height) # Size of the frames
fps = frame_rate # Frame rate of the video
out = cv2.VideoWriter(videopath, fourcc, fps, frame_size)
masks=[]
for frame in frames:
    # Apply the MOG2 algorithm to detect the moving objects
    mask = mog2.apply(frame)
    # Convert to BGR the mask to store it in the video
    mask = cv2.cvtColor(mask, cv2.COLOR_GRAY2BGR)
    masks.append(mask)
    # Save the mask in a video
    cv2.imshow("Moving Objects Mask", mask)
    if cv2.waitKey(1) & 0xFF == ord('q'):
        break
    out.write(mask)
cv2.imshow("Moving Objects Mask", masks[200])
cv2.waitKey(0)
out.release()
cv2.destroyAllWindows()

**Preguntas del Apartado C**
1. ¿Cómo afecta la variable `varThreshold` a la precisión de la detección?
2. ¿Qué ventajas presenta `createBackgroundSubtractorMOG2` frente a métodos simples de diferencia de imágenes?

## Apartado D: Flujo Óptico

### Tarea D.1: Configuración del Flujo Óptico

Consulta la documentación de cv2.calcOpticalFlowPyrLK para ver que parametros se deben definir para realizar el calculo del flujo óptico

In [7]:
#TODO: Use VideoCapture from OpenCV to read the video (slow_traffic_small.mp4)
videopath = "../data/partC/slow_traffic_small.mp4"  # Path to the video file

frames, frame_width, frame_height, frame_rate = read_video(videopath)
print(f"Número de frames: {len(frames)}")
print(f"Resolución: {frame_width}x{frame_height}")
print(f"Frame rate: {frame_rate}")


# Define the parameters for Lucas-Kanade optical flow
winSize = (15, 15)   
maxLevel = 2          
criteria = (cv2.TERM_CRITERIA_EPS | cv2.TERM_CRITERIA_COUNT, 10, 0.03) 

Número de frames: 914
Resolución: 640x360
Frame rate: 29.97002997002997


### Tarea D.2: Detección de Puntos de Interés

Detecta los puntos de interés iniciales usando el algoritmo de Shi-Tomasi (cv2.goodFeaturesToTrack) en el primer frame.

In [8]:
#TODO: Detect the initial points of interest in the first frame

# Convert the first frame to grayscale
prev_gray = cv2.cvtColor(frames[0], cv2.COLOR_BGR2GRAY)

# Define the parameters of the Shi-Tomasi algorithm
mask = None
maxCorners = 100
qualityLevel = 0.2
minDistance = 7
blockSize = 7

# Use the function goodFeaturesToTrack to detect the points of interest
p0 = cv2.goodFeaturesToTrack(prev_gray, mask=mask, maxCorners=maxCorners, qualityLevel=qualityLevel, minDistance=minDistance, blockSize=blockSize)

### Tarea D.3: Cálculo y Visualización del Flujo Óptico

In [9]:
#TODO: Use a loop to track the points of interest in the rest of the frames

# Create a mask image for drawing purposes
mask = np.zeros_like(frame)

for i, frame in enumerate(frames[1:]):
    input_frame = frame
    # Convert the frame to grayscale
    frame_gray = cv2.cvtColor(input_frame, cv2.COLOR_BGR2GRAY)
    # Calculate the optical flow using the Lucas-Kanade algorithm

    p1, st, err = cv2.calcOpticalFlowPyrLK(prev_gray, frame_gray, p0, None, winSize=winSize, maxLevel=maxLevel, criteria=criteria)

    # Select the points that were successfully tracked
    good_new = p1[st == 1]
    good_old = p0[st == 1]

    # Draw the tracks
    for i, (new, old) in enumerate(zip(good_new, good_old)):
        a, b = new.ravel().astype(int)
        c, d = old.ravel().astype(int)
        input_frame = cv2.circle(input_frame, (a, b), 5, (0, 0, 255), -1)
        mask = cv2.line(mask, (a, b), (c, d), (0, 255, 0), 2)

    # Update the inputs for the next iteration
    prev_gray = frame_gray.copy()
    p0 = good_new.reshape(-1, 1, 2)
    
    # Show the frame with the tracks
    cv2.imshow('Frame', cv2.add(input_frame, mask))
    key = cv2.waitKey(1)
    if key == ord('q'):
        break

cv2.destroyAllWindows()

**Preguntas del Apartado D**
1. ¿Qué efecto tiene el parámetro `winSize` en la precisión del flujo óptico?
2. ¿Cómo influye el parámetro `qualityLevel` en la función `cv2.goodFeaturesToTrack` al detectar puntos de interés?

PREGUNTA D.1 

In [10]:
# Mismo codigo que al anterior pero cambiando el winSize: 

prev_gray = cv2.cvtColor(frames[0], cv2.COLOR_BGR2GRAY)

# Define the parameters of the Shi-Tomasi algorithm
mask = None
maxCorners = 100
qualityLevel = 0.2
minDistance = 7
blockSize = 7

# Use the function goodFeaturesToTrack to detect the points of interest
p0 = cv2.goodFeaturesToTrack(prev_gray, mask=mask, maxCorners=maxCorners, qualityLevel=qualityLevel, minDistance=minDistance, blockSize=blockSize)
winSize = (5, 5)
# Create a mask image for drawing purposes


mask = np.zeros_like(frame)

for i, frame in enumerate(frames[1:]):
    input_frame = frame
    # Convert the frame to grayscale
    frame_gray = cv2.cvtColor(input_frame, cv2.COLOR_BGR2GRAY)
    # Calculate the optical flow using the Lucas-Kanade algorithm

    p1, st, err = cv2.calcOpticalFlowPyrLK(prev_gray, frame_gray, p0, None, winSize=winSize, maxLevel=maxLevel, criteria=criteria)

    # Select the points that were successfully tracked
    good_new = p1[st == 1]
    good_old = p0[st == 1]

    # Draw the tracks
    for i, (new, old) in enumerate(zip(good_new, good_old)):
        a, b = new.ravel().astype(int)
        c, d = old.ravel().astype(int)
        input_frame = cv2.circle(input_frame, (a, b), 5, (0, 0, 255), -1)
        mask = cv2.line(mask, (a, b), (c, d), (0, 255, 0), 2)

    # Update the inputs for the next iteration
    prev_gray = frame_gray.copy()
    p0 = good_new.reshape(-1, 1, 2)
    
    # Show the frame with the tracks
    cv2.imshow('Frame', cv2.add(input_frame, mask))
    key = cv2.waitKey(1)
    if key == ord('q'):
        break

cv2.destroyAllWindows()

In [11]:
# Mismo codigo que al anterior pero cambiando el winSize: 

prev_gray = cv2.cvtColor(frames[0], cv2.COLOR_BGR2GRAY)

# Define the parameters of the Shi-Tomasi algorithm
mask = None
maxCorners = 100
qualityLevel = 0.2
minDistance = 7
blockSize = 7

# Use the function goodFeaturesToTrack to detect the points of interest
p0 = cv2.goodFeaturesToTrack(prev_gray, mask=mask, maxCorners=maxCorners, qualityLevel=qualityLevel, minDistance=minDistance, blockSize=blockSize)
winSize = (30, 30)
# Create a mask image for drawing purposes


mask = np.zeros_like(frame)

for i, frame in enumerate(frames[1:]):
    input_frame = frame
    # Convert the frame to grayscale
    frame_gray = cv2.cvtColor(input_frame, cv2.COLOR_BGR2GRAY)
    # Calculate the optical flow using the Lucas-Kanade algorithm

    p1, st, err = cv2.calcOpticalFlowPyrLK(prev_gray, frame_gray, p0, None, winSize=winSize, maxLevel=maxLevel, criteria=criteria)

    # Select the points that were successfully tracked
    good_new = p1[st == 1]
    good_old = p0[st == 1]

    # Draw the tracks
    for i, (new, old) in enumerate(zip(good_new, good_old)):
        a, b = new.ravel().astype(int)
        c, d = old.ravel().astype(int)
        input_frame = cv2.circle(input_frame, (a, b), 5, (0, 0, 255), -1)
        mask = cv2.line(mask, (a, b), (c, d), (0, 255, 0), 2)

    # Update the inputs for the next iteration
    prev_gray = frame_gray.copy()
    p0 = good_new.reshape(-1, 1, 2)
    
    # Show the frame with the tracks
    cv2.imshow('Frame', cv2.add(input_frame, mask))
    key = cv2.waitKey(1)
    if key == ord('q'):
        break

cv2.destroyAllWindows()

PREGUNTA 2: 

In [12]:

# Mismo codigo que al anterior pero cambiando el quality level 

prev_gray = cv2.cvtColor(frames[0], cv2.COLOR_BGR2GRAY)

# Define the parameters of the Shi-Tomasi algorithm
mask = None
maxCorners = 100
qualityLevel = 0.05#muy bajo 
minDistance = 7
blockSize = 7

# Use the function goodFeaturesToTrack to detect the points of interest
p0 = cv2.goodFeaturesToTrack(prev_gray, mask=mask, maxCorners=maxCorners, qualityLevel=qualityLevel, minDistance=minDistance, blockSize=blockSize)
winSize = (10, 10)
# Create a mask image for drawing purposes


mask = np.zeros_like(frame)

for i, frame in enumerate(frames[1:]):
    input_frame = frame
    # Convert the frame to grayscale
    frame_gray = cv2.cvtColor(input_frame, cv2.COLOR_BGR2GRAY)
    # Calculate the optical flow using the Lucas-Kanade algorithm

    p1, st, err = cv2.calcOpticalFlowPyrLK(prev_gray, frame_gray, p0, None, winSize=winSize, maxLevel=maxLevel, criteria=criteria)

    # Select the points that were successfully tracked
    good_new = p1[st == 1]
    good_old = p0[st == 1]

    # Draw the tracks
    for i, (new, old) in enumerate(zip(good_new, good_old)):
        a, b = new.ravel().astype(int)
        c, d = old.ravel().astype(int)
        input_frame = cv2.circle(input_frame, (a, b), 5, (0, 0, 255), -1)
        mask = cv2.line(mask, (a, b), (c, d), (0, 255, 0), 2)

    # Update the inputs for the next iteration
    prev_gray = frame_gray.copy()
    p0 = good_new.reshape(-1, 1, 2)
    
    # Show the frame with the tracks
    cv2.imshow('Frame', cv2.add(input_frame, mask))
    key = cv2.waitKey(1)
    if key == ord('q'):
        break

cv2.destroyAllWindows()

In [13]:

# Mismo codigo que al anterior pero cambiando el quality level 

prev_gray = cv2.cvtColor(frames[0], cv2.COLOR_BGR2GRAY)

# Define the parameters of the Shi-Tomasi algorithm
mask = None
maxCorners = 100
qualityLevel = 0.4 #muy alto
minDistance = 7
blockSize = 7

# Use the function goodFeaturesToTrack to detect the points of interest
p0 = cv2.goodFeaturesToTrack(prev_gray, mask=mask, maxCorners=maxCorners, qualityLevel=qualityLevel, minDistance=minDistance, blockSize=blockSize)
winSize = (10, 10)
# Create a mask image for drawing purposes


mask = np.zeros_like(frame)

for i, frame in enumerate(frames[1:]):
    input_frame = frame
    # Convert the frame to grayscale
    frame_gray = cv2.cvtColor(input_frame, cv2.COLOR_BGR2GRAY)
    # Calculate the optical flow using the Lucas-Kanade algorithm

    p1, st, err = cv2.calcOpticalFlowPyrLK(prev_gray, frame_gray, p0, None, winSize=winSize, maxLevel=maxLevel, criteria=criteria)

    # Select the points that were successfully tracked
    good_new = p1[st == 1]
    good_old = p0[st == 1]

    # Draw the tracks
    for i, (new, old) in enumerate(zip(good_new, good_old)):
        a, b = new.ravel().astype(int)
        c, d = old.ravel().astype(int)
        input_frame = cv2.circle(input_frame, (a, b), 5, (0, 0, 255), -1)
        mask = cv2.line(mask, (a, b), (c, d), (0, 255, 0), 2)

    # Update the inputs for the next iteration
    prev_gray = frame_gray.copy()
    p0 = good_new.reshape(-1, 1, 2)
    
    # Show the frame with the tracks
    cv2.imshow('Frame', cv2.add(input_frame, mask))
    key = cv2.waitKey(1)
    if key == ord('q'):
        break

cv2.destroyAllWindows()

## Apartado E: Filtro de Kalman para Seguimiento de Objetos

### Tarea E.1: Configuración del Filtro de Kalman

Inicializa el filtro de Kalman (cv2.KalmanFilter) con una matriz de medición y transición adecuada para un seguimiento en dos dimensiones.

In [14]:
#TODO: Use VideoCapture from OpenCV to read the video (slow_traffic_small.mp4)
videopath = "../data/partC/slow_traffic_small.mp4"  # Path to the video file
cap = cv2.VideoCapture(videopath)  # Complete this line to read the video file

#TODO: Check if the video was successfully opened
if not cap.isOpened():
    print('Error: Could not open the video file')

#TODO: Get the szie of frames and the frame rate of the video
frame_width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
frame_height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
frame_rate = cap.get(cv2.CAP_PROP_FPS)

#TODO: Use a loop to read the frames of the video and store them in a list
frames = []
while True:
    ret, frame = cap.read()
    if not ret:
        break
    frames.append(frame)


cap.release()


# TODO: Create the Kalman filter object
n_states = 4 # medimos x, y, vx, vy
n_measurements = 2 # vemos solo x e y 
kf = cv2.KalmanFilter(n_states,n_measurements)

# TODO: Initialize the state of the Kalman filter
kf.measurementMatrix = np.array([
    [1, 0, 0, 0],   # medimos x
    [0, 1, 0, 0]    # medimos y
], dtype=np.float32)
# Measurement matrix np.array of shape (2, 4) and type np.float32

diferencia = 1.0/frame_rate

kf.transitionMatrix = kf.transitionMatrix = np.array([
    [1, 0, diferencia, 0],   # x' = x + vx
    [0, 1, 0, diferencia],   # y' = y + vy
    [0, 0, 1, 0],   # vx' = vx
    [0, 0, 0, 1]    # vy' = vy
], dtype=np.float32)


# Transition matrix np.array of shape (4, 4) and type np.float32
kf.processNoiseCov = np.eye(4, dtype=np.float32) * 1e-2 # Process noise covariance np.array of shape (4, 4) and type np.float32

measurement = np.zeros((2, 1), np.float32) 
prediction = np.zeros((4, 1), np.float32) 

#TODO: Show the frames to select the initial position of the object

for i, frame in enumerate(frames):
    input_frame = frame.copy()
    # Show the frame
    cv2.imshow('Frame', frame)
    # Wait for the key
    key = cv2.waitKey(0)

    # If the key is 'n' continue to the next frame
    if key == ord('n'):
        continue

    # If the key is 's' select the position of the object
    elif key == ord('s'):
        # Select the position of the object
        x, y, w, h = cv2.selectROI('Frame', frame, False)
        track_window = (x, y, w, h)

        #TODO: Compute the center of the object
        cx = x + w / 2.0
        cy = y + h / 2.0

        #TODO: Initialize the state of the Kalman filter
        kf.statePost = np.array([[cx], [cy], [0], [0]], np.float32)
        
        # Initialize the covariance matrix
        kf.errorCovPost = np.eye(4, dtype=np.float32)
        
        #Predict the position of the object
        prediction = kf.predict()

        #TODO: Update the measurement and correct the Kalman filter
        measurement = np.array([[cx], [cy]], np.float32)
        kf.correct(measurement)

        #TODO: Crop the object
        crop = frame[y:y+h, x:x+w].copy()
        # lo pasamos a HSV
        hsv_crop = cv2.cvtColor(crop, cv2.COLOR_BGR2HSV)
        
        #TODO: Compute the histogram of the cropped object (Reminder: Use only the Hue channel (0-180))
        mask_crop = cv2.inRange(hsv_crop, (0, 60, 32), (180, 255, 255))

        crop_hist = cv2.calcHist([hsv_crop], 
                                 [0], 
                                 #mask=None, 
                                 mask = mask_crop,
                                 histSize=[180], 
                                 ranges=[0, 180])
        cv2.normalize(crop_hist, crop_hist, 0, 255, cv2.NORM_MINMAX)
       
        print(f'Initial position selected: {x}, {y}')

        break

cv2.destroyAllWindows()

Initial position selected: 217, 84


### Tarea E.2: Predicción y Corrección del Estado

Realiza la predicción del estado y corrige la posición estimada en cada iteración.

In [15]:
#TODO: Use the Kalman filter to predict the position of the points of interest

term_crit = (cv2.TERM_CRITERIA_EPS | cv2.TERM_CRITERIA_COUNT, 30, 1)

for frame in frames[i+1:]:
    #TODO: Copy the frame 

    input_frame = frame.copy()
    #TODO: Convert the frame to HSV
    img_hsv = cv2.cvtColor(input_frame, cv2.COLOR_BGR2HSV)
    
   
    # Compute the back projection of the histogram
    mask_sv = cv2.inRange(img_hsv, (0, 60, 32), (180, 255, 255))
    img_bproject = cv2.calcBackProject([img_hsv], [0], crop_hist, [0, 180], 1)
    img_bproject &= mask_sv
    img_bproject = cv2.GaussianBlur(img_bproject, (7,7), 0)
   

    # Apply the mean shift algorithm to the back projection
    ret, track_window = cv2.meanShift(img_bproject, track_window, term_crit)
    x_,y_,w_,h_ = track_window
    #TODO: Compute the center of the object
    c_x = x_ + w_ / 2.0
    c_y = y_ + h_ / 2.0
    
    # Predict the position of the object
    prediction = kf.predict()

    #TODO: Update the measurement and correct the Kalman filter
    measurement = np.array([[c_x], [c_y]], np.float32)
    kf.correct(measurement)

    
    # Draw the predicted position
    cv2.circle(input_frame, (int(prediction[0][0]), int(prediction[1][0])), 5, (0, 0, 255), -1)
    cv2.circle(input_frame, (int(c_x), int(c_y)), 5, (0, 255, 0), -1)
    cv2.rectangle(input_frame, (x_, y_), (x_ + w_, y_ + h_), (255, 0, 0), 2)

    # Show the frame with the predicted position
    cv2.imshow('Frame', input_frame)
    key = cv2.waitKey(30) & 0xFF
    if key == ord('q'):
        break

cv2.destroyAllWindows()


**Preguntas del Apartado E**
1. ¿Cómo afecta el valor de `transitionMatrix` a la predicción en el filtro de Kalman?
2. ¿Cuál es la diferencia entre `measurementMatrix` y `transitionMatrix` en el contexto del seguimiento de objetos?

PREGUNTA 1: Efecto de transitionMatrix

In [25]:
import cv2
import numpy as np

# ============================
#   ELIGE EL CASO A PROBAR
# ============================
# "A1"    -> identidad (modelo SIN velocidad)
# "A_bad" -> velocidad invertida (modelo malísimo)
CASE = "A1"   # <-- CAMBIA A "A_bad" PARA EL OTRO CASO


# ============================
#   LEER VIDEO
# ============================
videopath = "../data/partC/slow_traffic_small.mp4"
cap = cv2.VideoCapture(videopath)

if not cap.isOpened():
    raise ValueError("Error: Could not open the video file")

frame_width  = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
frame_height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
frame_rate   = cap.get(cv2.CAP_PROP_FPS)
dt = 1.0 / frame_rate

frames = []
while True:
    ret, frame = cap.read()
    if not ret:
        break
    frames.append(frame)
cap.release()


# ============================
#   CREAR KALMAN
# ============================
kf = cv2.KalmanFilter(4, 2)

kf.measurementMatrix = np.array([
    [1, 0, 0, 0],
    [0, 1, 0, 0]
], dtype=np.float32)

# --- Matrices de transición ---
A1 = np.array([  # Identidad: NO usa velocidad para mover posición
    [1, 0, 0, 0],
    [0, 1, 0, 0],
    [0, 0, 1, 0],
    [0, 0, 0, 1]
], dtype=np.float32)

A_bad = np.array([  # Velocidad invertida: predice en dirección contraria
    [1, 0, -dt, 0],
    [0, 1, 0, -dt],
    [0, 0,  1, 0],
    [0, 0,  0, 1]
], dtype=np.float32)

kf.transitionMatrix = A1 

kf.processNoiseCov = np.eye(4, dtype=np.float32) * 1e-2
kf.measurementNoiseCov = np.eye(2, dtype=np.float32) * 1e-1

measurement = np.zeros((2,1), np.float32)
prediction  = np.zeros((4,1), np.float32)


# ============================
#   SELECCIÓN ROI + HISTO
# ============================
for i, frame in enumerate(frames):
    cv2.imshow("Frame", frame)
    key = cv2.waitKey(0) & 0xFF

    if key == ord('n'):
        continue
    elif key == ord('s'):
        cv2.destroyWindow("Frame")
        x, y, w, h = cv2.selectROI("ROI", frame, False)
        cv2.destroyWindow("ROI")

        track_window = (x, y, w, h)

        cx = x + w/2.0
        cy = y + h/2.0

        # Fuerzo velocidad inicial para que el efecto de A se vea sí o sí
        kf.statePost = np.array([[cx], [cy], [8], [8]], np.float32)
        kf.errorCovPost = np.eye(4, dtype=np.float32)

        crop = frame[y:y+h, x:x+w].copy()
        hsv_crop = cv2.cvtColor(crop, cv2.COLOR_BGR2HSV)
        mask_crop = cv2.inRange(hsv_crop, (0,60,32), (180,255,255))

        crop_hist = cv2.calcHist([hsv_crop], [0], mask_crop, [180], [0,180])
        cv2.normalize(crop_hist, crop_hist, 0, 255, cv2.NORM_MINMAX)

        print("ROI inicial:", x, y)
        break

cv2.destroyAllWindows()


# ============================
#   TRACKING + KALMAN (E3)
# ============================
term_crit = (cv2.TERM_CRITERIA_EPS | cv2.TERM_CRITERIA_COUNT, 30, 1)

for frame in frames[i+1:]:
    input_frame = frame.copy()
    img_hsv = cv2.cvtColor(input_frame, cv2.COLOR_BGR2HSV)

    mask_sv = cv2.inRange(img_hsv, (0,60,32), (180,255,255))
    img_bproject = cv2.calcBackProject([img_hsv], [0], crop_hist, [0,180], 1)
    img_bproject &= mask_sv
    img_bproject = cv2.GaussianBlur(img_bproject, (7,7), 0)

    ret, track_window = cv2.meanShift(img_bproject, track_window, term_crit)
    x_, y_, w_, h_ = track_window
    c_x = x_ + w_/2.0
    c_y = y_ + h_/2.0

    # Predict + Correct
    prediction = kf.predict()
    measurement = np.array([[c_x], [c_y]], np.float32)
    kf.correct(measurement)

    # Dibujo (rojo = predicción, verde = medición)
    cv2.circle(input_frame, (int(prediction[0][0]), int(prediction[1][0])), 5, (0,0,255), -1)
    cv2.circle(input_frame, (int(c_x), int(c_y)), 5, (0,255,0), -1)
    cv2.rectangle(input_frame, (x_, y_), (x_ + w_, y_ + h_), (255,0,0), 2)

    cv2.putText(input_frame, f"CASE: {CASE}", (20,30),
                cv2.FONT_HERSHEY_SIMPLEX, 1, (255,255,255), 2)

    cv2.imshow("Tracking", input_frame)
    key = cv2.waitKey(30) & 0xFF
    if key == ord('q'):
        break

cv2.destroyAllWindows()


ROI inicial: 220 105
